In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
PROC = ROOT / 'datasets' / 'processed' / 'module3_uturn'
PROC.mkdir(parents=True, exist_ok=True)

labels = pd.read_csv(PROC / 'uturn_candidate_labels.csv')
tracks = pd.read_parquet(PROC / 'ngsim_tracks_sample.parquet')

print('labels', labels.shape, labels.columns.tolist())
print('tracks', tracks.shape, tracks.columns.tolist())
print(labels.head())
print('uturn_candidate counts:\n', labels['uturn_candidate'].value_counts())

Mounted at /content/drive
labels (1102, 5) ['vehicle_id', 'max_abs_turn', 'n_frames', 'mean_speed', 'uturn_candidate']
tracks (250000, 11) ['vehicle_id', 'frame_id', 'x', 'y', 'lane_id', 'speed', 'dx', 'dy', 'heading', 'heading_unwrap', 'heading_delta']
   vehicle_id  max_abs_turn  n_frames  mean_speed  uturn_candidate
0           2      2.623276        82   25.628659                1
1           7      3.078106        22   23.568182                1
2           8      2.254157         9   29.490000                0
3          11      3.141566       744    7.825242                1
4          13      3.140467        80   26.009875                1
uturn_candidate counts:
 uturn_candidate
1    613
0    489
Name: count, dtype: int64


In [2]:
# ensure expected columns
need_track = {'vehicle_id', 'frame_id', 'x', 'y'}
missing = need_track - set(tracks.columns)
assert not missing, f'track columns missing: {missing}'

tr = tracks.sort_values(['vehicle_id', 'frame_id']).copy()

# motion features
tr['dx'] = tr.groupby('vehicle_id')['x'].diff()
tr['dy'] = tr.groupby('vehicle_id')['y'].diff()
tr['step_dist'] = np.sqrt(tr['dx']**2 + tr['dy']**2)
tr['heading'] = np.arctan2(tr['dy'], tr['dx'])
tr['heading_u'] = tr.groupby('vehicle_id')['heading'].transform(lambda s: np.unwrap(s.fillna(0).to_numpy()))
tr['dheading'] = tr.groupby('vehicle_id')['heading_u'].diff()

agg = tr.groupby('vehicle_id').agg(
    n_frames=('frame_id', 'count'),
    duration_frames=('frame_id', lambda s: s.max() - s.min() + 1),
    x_range=('x', lambda s: s.max() - s.min()),
    y_range=('y', lambda s: s.max() - s.min()),
    mean_step=('step_dist', 'mean'),
    std_step=('step_dist', 'std'),
    max_step=('step_dist', 'max'),
    mean_speed=('speed', 'mean') if 'speed' in tr.columns else ('step_dist', 'mean'),
    std_speed=('speed', 'std') if 'speed' in tr.columns else ('step_dist', 'std'),
    max_abs_dheading=('dheading', lambda s: np.nanmax(np.abs(s))),
    sum_abs_dheading=('dheading', lambda s: np.nansum(np.abs(s))),
    mean_abs_dheading=('dheading', lambda s: np.nanmean(np.abs(s))),
).reset_index()

# path curvature proxy
agg['path_length'] = tr.groupby('vehicle_id')['step_dist'].sum().values
agg['displacement'] = np.sqrt(agg['x_range']**2 + agg['y_range']**2)
agg['tortuosity'] = agg['path_length'] / (agg['displacement'] + 1e-6)
agg['turn_intensity'] = agg['sum_abs_dheading'] / (agg['n_frames'] + 1e-6)

# join label
lab = labels.copy()
if 'uturn_candidate' not in lab.columns:
    # fallback name
    cand_col = [c for c in lab.columns if 'uturn' in c.lower() or 'candidate' in c.lower()][0]
    lab = lab.rename(columns={cand_col: 'uturn_candidate'})

ml = agg.merge(lab[['vehicle_id', 'uturn_candidate']], on='vehicle_id', how='inner')
ml['y'] = ml['uturn_candidate'].astype(int)

print('ML rows', len(ml))
print(ml['y'].value_counts())
print(ml.head())

ML rows 1102
y
1    613
0    489
Name: count, dtype: int64
   vehicle_id  n_frames  duration_frames  x_range  y_range  mean_step  \
0           2        82               82   47.443  183.214   2.709935   
1           7        22               22   79.704   16.706   3.892768   
2           8         9                9   12.432   16.964   2.802924   
3          11       744              673   16.914  298.540  13.624395   
4          13        80               80   34.850  179.822   2.571912   

    std_step    max_step  mean_speed  std_speed  max_abs_dheading  \
0   0.902830    4.331984   25.628659  11.079938          2.623276   
1   1.283057    7.501053   23.568182   1.027756          3.078106   
2   0.459241    3.297419   29.490000   0.000000          2.254157   
3  32.330330  141.120419    7.825242  13.420361          3.141566   
4   0.994150    4.471689   26.009875   9.330376          3.140467   

   sum_abs_dheading  mean_abs_dheading   path_length  displacement  \
0          6.9745

In [3]:
feat_cols = [
    'n_frames', 'duration_frames',
    'x_range', 'y_range',
    'mean_step', 'std_step', 'max_step',
    'mean_speed', 'std_speed',
    'max_abs_dheading', 'sum_abs_dheading', 'mean_abs_dheading',
    'path_length', 'displacement', 'tortuosity', 'turn_intensity'
]

for c in feat_cols:
    if c not in ml.columns:
        ml[c] = 0.0

Xdf = ml[feat_cols + ['y', 'vehicle_id']].copy()
Xdf[feat_cols] = Xdf[feat_cols].replace([np.inf, -np.inf], np.nan)
Xdf[feat_cols] = Xdf[feat_cols].fillna(Xdf[feat_cols].median(numeric_only=True))

# drop tiny tracks
Xdf = Xdf[Xdf['n_frames'] >= 10].copy()

print('final', Xdf.shape)
print(Xdf['y'].value_counts())

out_parquet = PROC / 'module3_ml_table.parquet'
out_csv = PROC / 'module3_ml_table_sample.csv'
Xdf.to_parquet(out_parquet, index=False)
Xdf.to_csv(out_csv, index=False)

meta = {
    'rows': int(len(Xdf)),
    'positives': int(Xdf['y'].sum()),
    'negatives': int((Xdf['y'] == 0).sum()),
    'features': feat_cols,
    'label_source': 'NGSIM heading-reversal heuristic (uturn_candidate)',
    'feature_source': 'NGSIM trajectory geometry/kinematics',
    'note': 'U-turn candidates from abs heading change; refine later with SUMO/inD if available'
}
(PROC / 'module3_ml_meta.json').write_text(json.dumps(meta, indent=2))

# verify on disk immediately
print('saved', out_parquet.exists(), out_parquet.stat().st_size)
print(json.dumps(meta, indent=2))
ok = len(Xdf) > 100 and Xdf['y'].nunique() == 2
print('STEP 8 OK' if ok else 'STEP 8 NEED CHECK — class balance/rows')

final (1097, 18)
y
1    612
0    485
Name: count, dtype: int64
saved True 159932
{
  "rows": 1097,
  "positives": 612,
  "negatives": 485,
  "features": [
    "n_frames",
    "duration_frames",
    "x_range",
    "y_range",
    "mean_step",
    "std_step",
    "max_step",
    "mean_speed",
    "std_speed",
    "max_abs_dheading",
    "sum_abs_dheading",
    "mean_abs_dheading",
    "path_length",
    "displacement",
    "tortuosity",
    "turn_intensity"
  ],
  "label_source": "NGSIM heading-reversal heuristic (uturn_candidate)",
  "feature_source": "NGSIM trajectory geometry/kinematics",
  "note": "U-turn candidates from abs heading change; refine later with SUMO/inD if available"
}
STEP 8 OK
